[![Open Rendered Output](https://img.shields.io/badge/Rendered%20Output-Open-blue?logo=link&logoColor=white)](https://htmlpreview.github.io/?https://raw.githubusercontent.com/MeteoSwiss/nwp-fdb-polytope-demo/main/examples/snapshots/swiss_lv95_regridding.html)

# Regrid a full ICON-CH1-EPS field to the Swiss LV95 grid

This example retrieves one **full ICON-CH1-EPS field** from MeteoSwiss Polytope and interpolates it to the predefined **Swiss LV95 1-km grid**.


<div style="text-align:center;">
  <img src="https://raw.githubusercontent.com/MeteoSwiss/nwp-fdb-polytope-demo/main/examples/Polytope/images/icon_ch1_eps_comparison.png
  " style="width:70%;"/>
</div>

## Environment setup

Create a clean Python 3.11 environment:

```bash
python3.11 -m venv .venv-swisslv95-dev214
source .venv-swisslv95-dev214/bin/activate

python -m pip install --upgrade pip setuptools wheel

python -m pip install \
    "earthkit-data[polytope]==1.1.0" \
    earthkit-geo==1.1.2 \
    eccodes-cosmo-resources-python==2.47.0.1 \
    ipykernel
```

For this example, the current Swiss LV95 fix is tested with temporary ECMWF `dev214` wheels for the MIR/eckit backend:

```bash
python -m pip install \
    -i https://get-test.ecmwf.int/repository/pypi-betas/simple/ \
    --extra-index-url https://pypi.org/simple \
    eckit==2.4.1.dev214 \
    mir-python==1.30.3.dev214
```


> **Note:**
> - Compatibility warnings from `pip` are expected because the temporary `dev214` backend wheels override versions pinned by the current Earthkit releases.
> - These wheels are temporary and used to validate Swiss LV95 support.
> - The additional beta-wheel installation should no longer be needed once the corresponding earthkit-geo release is available.


Register the environment as a Jupyter kernel:

```bash
python -m ipykernel install --user \
    --name swisslv95-dev214 \
    --display-name "Python (Swiss LV95 dev214)"
```


### Check the main package versions

The cell below prints the main packages relevant to this example so the environment used for the interpolation is easy to reproduce.

In [1]:
from importlib.metadata import PackageNotFoundError, version

packages = [
    "earthkit-data",
    "earthkit-geo",
    "eckit",
    "mir-python",
    "eccodes-cosmo-resources-python",
]

for package in packages:
    try:
        print(f"{package:<32}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:<32}: NOT INSTALLED")

earthkit-data                   : 1.1.0
earthkit-geo                    : 1.1.2
eckit                           : 2.4.1.dev214
mir-python                      : 1.30.3.dev214
eccodes-cosmo-resources-python  : 2.47.0.1


## Configure Polytope access

To access ICON data via MeteoSwiss Polytope, create a `config.yml` based on the repository's `config_example.yml` and add your MeteoSwiss offline token.


In [2]:
import os
import yaml

def load_config(path="config.yml"):
    if not os.path.exists(path):
        raise FileNotFoundError(
            "Missing config.yml. Please create one based on config_example.yml."
        )
    with open(path, "r") as f:
        return yaml.safe_load(f)

config = load_config()

os.environ["POLYTOPE_USER_KEY"] = config["meteoswiss"]["key"]
os.environ["POLYTOPE_ADDRESS"] = "https://service.meteoswiss.ch/polytope"


## Configure MeteoSwiss ecCodes definitions

MeteoSwiss uses additional ecCodes definitions for ICON data. Configure them before reading the GRIB field.


In [3]:
import os

os.environ["ECCODES_ECKIT_GEO"] = "1"
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

import eccodes
import eccodes_cosmo_resources

vendor = eccodes.codes_definition_path()
cosmo = eccodes_cosmo_resources.get_definitions_path()
eccodes.codes_set_definitions_path(f"{cosmo}:{vendor}")


## Choose a recent model run automatically

Use a time approximately 12 hours in the past and round it down to the previous 3-hour model cycle. This avoids having to edit the date manually each time the notebook is run.


In [4]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)
past_time = now - timedelta(hours=12)

rounded_hour = (past_time.hour // 3) * 3
rounded_time = past_time.replace(
    hour=rounded_hour,
    minute=0,
    second=0,
    microsecond=0,
)

request_date = rounded_time.strftime("%Y%m%d")
request_time = rounded_time.strftime("%H%M")

print("Model cycle:", request_date, request_time)


Model cycle: 20261008 1500


## Retrieve one full ICON-CH1-EPS field

The `mch` collection returns the full field. Here we request 2-metre temperature (`T_2M`) as a simple example.


In [5]:
import earthkit.data as ekd

request = {
    "param": "500011",       # T_2M
    "date": request_date,
    "time": request_time,
    "expver": "0001",
    "step": 0,
    "class": "od",
    "levtype": "sfc",
    "model": "icon-ch1-eps",
    "stream": "enfo",
    "type": "CF",
    "timespan": "none",
}

ds = ekd.from_source(
    "polytope",
    "mch",
    request,
    stream=False,
)

field = ds.to_fieldlist()[0]
field


2026-10-09 07:45:42 - INFO - Sending request...
{'request': 'class: od\n'
            "date: '20261008'\n"
            "expver: '0001'\n"
            'levtype: sfc\n'
            'model: icon-ch1-eps\n'
            "param: '500011'\n"
            'step: 0\n'
            'stream: enfo\n'
            "time: '1500'\n"
            'timespan: none\n'
            'type: CF\n',
 'verb': 'retrieve'}
2026-10-09 07:45:43 - INFO - Request accepted. Please poll ./29445deb-5fad-414b-a2c9-4adc74690427 for status
2026-10-09 07:45:43 - INFO - Checking request status (29445deb-5fad-414b-a2c9-4adc74690427)...
2026-10-09 07:45:43 - INFO - The current status of the request is 'queued'
2026-10-09 07:45:43 - INFO - The current status of the request is 'processing'
2026-10-09 07:45:45 - INFO - The current status of the request is 'processed'
                                                                                                                      

number_of_values,1147980
array_type,ndarray
array_dtype,float64
variable,T_2M
standard_name,air_temperature
long_name,2m Temperature
units,kelvin
valid_datetime,2026-10-08 15:00:00
base_datetime,2026-10-08 15:00:00
step,0:00:00
level,2


## Inspect the native ICON grid


In [6]:
print(field.ls())
print("gridType:", field.metadata("gridType"))
print("grid spec:", field.geography.grid_spec())

values = field.to_numpy()
print("number of source points:", values.size)


  parameter.variable time.valid_datetime  time.base_datetime time.step  \
0               T_2M 2026-10-08 15:00:00 2026-10-08 15:00:00    0 days   

   vertical.level        vertical.level_type ensemble.member  \
0               2  height_above_ground_level               0   

  geography.grid_type  
0                ICON  
gridType: unstructured_grid
grid spec: {'area': [50.6, -0.9, 41.9, 17.8], 'grid': 'ICON-CH1_C', 'uid': '17643da2574959b644d254a3cd6e2bc0'}
number of source points: 1147980


## Swiss LV95 target grid

**LV95** is Switzerland's national projected coordinate system, **CH1903+ / LV95 (EPSG:2056)**.

Unlike latitude/longitude coordinates, LV95 uses projected **eastings and northings in metres**. The predefined `swisslv95` grid is a regular 1-km grid covering Switzerland.

For the standard MeteoSwiss grid, the easiest form is simply:

```python
Grid("swisslv95")
```

Its 1-km grid corresponds to the following explicit specification:

```python
{
    "type": "swisslv95",
    "x": [2439000, 2867000, 1000],
    "y": [1040500, 1333500, 1000],
}
```

The explicit form is useful when a different LV95 extent or spacing is required.


In [7]:
from eckit.geo import Grid

target_grid = Grid("swisslv95")

print("Swiss LV95 grid specification:")
print(target_grid.spec)
print("shape:", target_grid.shape)
print("size:", target_grid.size())


Swiss LV95 grid specification:
{'area': [48.1517247240718, 5.27585865137911, 45.4640456795421, 11.023684403413], 'bounding_box_xy': [2439000, 1040500, 2867000, 1333500], 'figure': {'a': 6377397.155, 'b': 6356078.96281819}, 'grid': [1000, 1000], 'order': 'i+j+', 'projection': {'target': 'EPSG:2056', 'type': 'proj'}, 'shape': [429, 294], 'type': 'regular_xy'}
shape: (294, 429)
size: 126126


## Regrid ICON-CH1-EPS to Swiss LV95

`earthkit-geo` delegates the interpolation to MIR. Here we use linear interpolation.

The first interpolation for a given source grid, target grid and interpolation method may take longer because MIR computes the interpolation coefficients (weights). MIR can cache these weights and reuse them for later fields on the same pair of grids.

The returned grid metadata is printed as well. With the updated backend, it should describe the Swiss LV95 target grid rather than the previous `unstructured_ll` representation.


In [8]:
import earthkit.geo as ekg

result, result_grid = ekg.grids.array.regrid(
    values,
    in_grid=field.geography.grid_spec(),
    out_grid=target_grid,
    interpolation="linear",
)

print("source points:", values.size)
print("target points:", target_grid.size())
print("output points:", result.size)

print("\nRequested Swiss LV95 target grid:")
print(target_grid.spec)

print("\nReturned result grid:")
print(result_grid)


Exception: Serious bug: Representation::croppedRepresentation() not implemented for RegularXY[{"area":[48.1517247240718,5.27585865137911,45.4640456795421,11.023684403413],"bounding_box_xy":[2439000,1040500,2867000,1333500],"figure":{"a":6377397.155,"b":6356078.96281819},"grid":[1000,1000],"order":"i+j+","projection":{"target":"EPSG:2056","type":"proj"},"shape":[429,294],"type":"regular_xy"}] 
Serious bug: Representation::croppedRepresentation() not implemented for RegularXY[{"area":[48.1517247240718,5.27585865137911,45.4640456795421,11.023684403413],"bounding_box_xy":[2439000,1040500,2867000,1333500],"figure":{"a":6377397.155,"b":6356078.96281819},"grid":[1000,1000],"order":"i+j+","projection":{"target":"EPSG:2056","type":"proj"},"shape":[429,294],"type":"regular_xy"}]
backtrace [2] stack has 66 addresses
(/scratch/mch/nburgdor/02_iwf/test_swiss_Grid/.venv-swisslv95-dev214/lib/python3.11/site-packages/eckitlib/lib64/../lib64/libeckit.so+eckit::BackTrace::dump[abi:cxx11]())0x43 
(/scrat

RuntimeError: Serious bug: Representation::croppedRepresentation() not implemented for RegularXY[{"area":[48.1517247240718,5.27585865137911,45.4640456795421,11.023684403413],"bounding_box_xy":[2439000,1040500,2867000,1333500],"figure":{"a":6377397.155,"b":6356078.96281819},"grid":[1000,1000],"order":"i+j+","projection":{"target":"EPSG:2056","type":"proj"},"shape":[429,294],"type":"regular_xy"}]

### Alternative: define the LV95 target explicitly

Instead of the predefined named grid, pass a grid specification directly:

```python
custom_target_grid = {
    "type": "swisslv95",
    "x": [2439000, 2867000, 1000],
    "y": [1040500, 1333500, 1000],
}

result, result_grid = ekg.grids.array.regrid(
    values,
    in_grid=field.geography.grid_spec(),
    out_grid=custom_target_grid,
    interpolation="linear",
)
```

Change the x/y bounds or spacing when a different LV95 target domain is needed.


## Sanity checks

The source ICON grid contains many more points than the 1-km Swiss target grid. The output size should therefore match the Swiss LV95 target grid rather than the number of source points.

The value range can also change slightly after interpolation. This is expected: the target grid samples different locations, and linear interpolation computes each target value from neighbouring source values instead of copying source grid points directly. As a result, source-grid extrema do not necessarily appear at exactly the same values on the target grid.

The checks below compare the number of points, value ranges and missing values.


In [ ]:
import numpy as np

print(f"Source points: {values.size}")
print(f"Target points: {target_grid.size()}")
print(f"Output points: {result.size}")

print(
    f"Source range: {np.nanmin(values):.3f} to {np.nanmax(values):.3f}"
)
print(
    f"Result range: {np.nanmin(result):.3f} to {np.nanmax(result):.3f}"
)

print(f"Source NaNs: {np.isnan(values).sum()}")
print(f"Result NaNs: {np.isnan(result).sum()}")

print("\nReturned result grid:")
print(result_grid)

assert result.size == target_grid.size()


## Notes on interpolation weights

When `earthkit-geo` performs the regridding, it uses [MIR](https://www.ecmwf.int/en/newsletter/152/computing/new-ecmwf-interpolation-package-mir) (Meteorological Interpolation and Regridding), ECMWF's interpolation library, under the hood.

For each combination of:
- source grid,
- target grid,
- interpolation method,

MIR computes interpolation coefficients. These coefficients can be cached and reused for repeated interpolations on the same grids, which makes subsequent runs faster.

The cache location depends on the system configuration. On some systems it may use a temporary directory such as `/tmp`, so the cache may be recreated when the environment or machine changes.
